In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_6.py ──
"""Shared infrastructure for MLFP02 Exercise 6 — Logistic Regression and
Classification Foundations.

Contains: HDB data loading, binary target construction, design matrix
preparation, sigmoid implementation, the neg-log-likelihood + gradient used
by scipy.optimize, calibration-curve binning, and output-directory setup.

Technique-specific code (odds ratios, threshold sweeps, confusion matrices,
ROC/PR/calibration plots, ANOVA + Tukey HSD) does NOT belong here — it
lives in the per-technique files in solutions/ex_6/ and local/ex_6/.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "mlfp02_ex6_logistic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS: list[str] = ["floor_area_sqm", "storey_mid", "remaining_lease"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — HDB resale transactions (Singapore)
# ════════════════════════════════════════════════════════════════════════


def load_hdb_recent() -> pl.DataFrame:
    """Load HDB resale transactions filtered to 2020+.

    Returns a polars DataFrame with the raw columns from the dataset
    (resale_price, flat_type, floor_area_sqm, storey_range, lease_commence_date,
    month, etc.). No target construction — see build_classification_frame().
    """
    loader = MLFPDataLoader()
    hdb = loader.load("mlfp01", "hdb_resale.parquet")
    return hdb.filter(pl.col("month").str.to_date("%Y-%m") >= pl.date(2020, 1, 1))


def build_classification_frame(hdb_recent: pl.DataFrame) -> tuple[pl.DataFrame, float]:
    """Add the binary target and engineered features used by logistic regression.

    The target ``high_price`` = 1 if resale_price > median, 0 otherwise — this
    gives a balanced ~50/50 split so the baseline accuracy is 50%.

    Returns (dataframe, median_price).
    """
    median_price = hdb_recent["resale_price"].median()
    frame = hdb_recent.with_columns(
        (pl.col("resale_price") > median_price).cast(pl.Int32).alias("high_price"),
        (
            (
                pl.col("storey_range").str.extract(r"(\d+)", 1).cast(pl.Float64)
                + pl.col("storey_range").str.extract(r"TO (\d+)", 1).cast(pl.Float64)
            )
            / 2.0
        ).alias("storey_mid"),
        (
            99
            - (
                pl.col("month").str.to_date("%Y-%m").dt.year()
                - pl.col("lease_commence_date")
            )
        )
        .cast(pl.Float64)
        .alias("remaining_lease"),
    ).drop_nulls(
        subset=["floor_area_sqm", "storey_mid", "high_price", "remaining_lease"]
    )
    return frame, float(median_price)


def build_design_matrix(
    frame: pl.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]:
    """Standardise features and append an intercept column.

    Returns (X_with_intercept, y, X_mean, X_std, feature_names_with_intercept).
    """
    X_raw = frame.select(FEATURE_COLS).to_numpy().astype(np.float64)
    y = frame["high_price"].to_numpy().astype(np.float64)

    X_mean = X_raw.mean(axis=0)
    X_std = X_raw.std(axis=0)
    X_scaled = (X_raw - X_mean) / X_std

    n_obs = X_scaled.shape[0]
    X = np.column_stack([np.ones(n_obs), X_scaled])
    feature_names = ["intercept", *FEATURE_COLS]
    return X, y, X_mean, X_std, feature_names


# ════════════════════════════════════════════════════════════════════════
# SIGMOID + BERNOULLI LOG-LIKELIHOOD
# ════════════════════════════════════════════════════════════════════════


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid σ(z) = 1 / (1 + exp(-z)).

    For z ≥ 0 use 1/(1+exp(-z)); for z < 0 use exp(z)/(1+exp(z)). Both
    branches avoid overflow at the extremes.
    """
    z = np.asarray(z, dtype=np.float64)
    result = np.zeros_like(z)
    pos = z >= 0
    neg = ~pos
    result[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    exp_z = np.exp(z[neg])
    result[neg] = exp_z / (1.0 + exp_z)
    return result


def neg_log_likelihood_logistic(
    beta: np.ndarray, X: np.ndarray, y: np.ndarray
) -> float:
    """Negative Bernoulli log-likelihood: -Σ[y log p + (1-y) log(1-p)]."""
    z = X @ beta
    p = sigmoid(z)
    p = np.clip(p, 1e-15, 1 - 1e-15)
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float(-ll)


def neg_ll_gradient(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Gradient of the negative log-likelihood: -X^T (y - p)."""
    z = X @ beta
    p = sigmoid(z)
    return -X.T @ (y - p)


# ════════════════════════════════════════════════════════════════════════
# ORIGINAL-SCALE COEFFICIENT CONVERSION
# ════════════════════════════════════════════════════════════════════════


def unscale_coefficients(
    beta_scaled: np.ndarray, X_mean: np.ndarray, X_std: np.ndarray
) -> np.ndarray:
    """Convert coefficients fit on standardised features to the original scale.

    β_original[j]   = β_scaled[j] / σ_j  for j ≥ 1
    β_original[0]   = β_scaled[0] - Σ β_scaled[j] * μ_j / σ_j
    """
    beta_original = np.zeros_like(beta_scaled)
    beta_original[0] = beta_scaled[0] - float(np.sum(beta_scaled[1:] * X_mean / X_std))
    beta_original[1:] = beta_scaled[1:] / X_std
    return beta_original


# ════════════════════════════════════════════════════════════════════════
# CALIBRATION BINNING
# ════════════════════════════════════════════════════════════════════════


def calibration_bins(
    y: np.ndarray, p: np.ndarray, n_bins: int = 10
) -> tuple[list[float], list[float], list[int]]:
    """Equal-width binning over predicted probabilities.

    Returns (mean_predicted, mean_observed, counts) for non-empty bins only.
    """
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    mean_pred: list[float] = []
    mean_obs: list[float] = []
    counts: list[int] = []
    for i in range(n_bins):
        hi = edges[i + 1] + (1e-12 if i == n_bins - 1 else 0.0)
        mask = (p >= edges[i]) & (p < hi)
        if mask.sum() > 0:
            mean_pred.append(float(p[mask].mean()))
            mean_obs.append(float(y[mask].mean()))
            counts.append(int(mask.sum()))
    return mean_pred, mean_obs, counts




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 6.4: Calibration Assessment and ANOVA
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Assess model calibration with calibration curves and Brier score
#   - Understand what "well-calibrated" means for decision-making
#   - Perform one-way ANOVA and interpret F-statistics + eta-squared
#   - Apply post-hoc Tukey HSD for pairwise group comparisons
#   - Apply calibration + ANOVA to Singapore HDB policy analysis
#
# PREREQUISITES: Exercises 6.1-6.3 (logistic regression, metrics)
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — calibration vs discrimination, ANOVA vs t-test
#   2. Build — calibration curve + Brier score
#   3. Train — one-way ANOVA + Tukey HSD across flat types
#   4. Visualise — calibration plot + ANOVA box plots
#   5. Apply — HDB flat-type pricing policy (S$ impact)
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
import plotly.graph_objects as go
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import brier_score_loss



## THEORY — Calibration vs Discrimination; ANOVA vs t-test


In [ ]:
# CALIBRATION: "When the model says P=0.7, do ~70% of those cases
# turn out positive?" Matters when you USE probabilities for decisions
# (pricing, insurance premiums), not just for ranking.
#
# DISCRIMINATION: "Can the model separate positives from negatives?"
# (measured by AUC). A model can discriminate well but be poorly
# calibrated — it ranks correctly but the probability values are wrong.
#
# BRIER SCORE = mean((p - y)^2). Lower = better. Random = 0.25.
#
# ANOVA generalises the t-test from 2 groups to k groups.
# H0: all group means are equal. H1: at least one differs.
# Tukey HSD runs pairwise comparisons with correction for multiple
# testing so the family-wise error rate stays at 5%.



## TASK 2 — BUILD: calibration curve + Brier score


In [ ]:
print("\n" + "=" * 70)
print("  Calibration Assessment and ANOVA")
print("=" * 70)

# Load data and fit logistic regression
hdb_recent = load_hdb_recent()
frame, median_price = build_classification_frame(hdb_recent)
X, y, X_mean, X_std, feature_names = build_design_matrix(frame)
n_obs = X.shape[0]
n_positive = int(y.sum())

beta0 = np.zeros(X.shape[1])
result = minimize(
    neg_log_likelihood_logistic,
    beta0,
    args=(X, y),
    method="L-BFGS-B",
    jac=neg_ll_gradient,
    options={"maxiter": 1000, "ftol": 1e-12},
)
beta_scratch = result.x
p_scratch = sigmoid(X @ beta_scratch)

# TODO: Compute the Brier score.
# Hint: brier_score_loss(y, p_scratch).
brier = ____

# TODO: Compute calibration bins using the shared helper.
# Hint: calibration_bins(y, p_scratch, n_bins=10) returns
#       (cal_predicted, cal_observed, cal_counts).
cal_predicted, cal_observed, cal_counts = ____

print(f"\n=== Model Calibration ===")
print(f"Brier score: {brier:.6f} (lower = better, 0 = perfect)")
print(f"Max Brier (random): 0.25")
print(f"Brier skill: {1 - brier / 0.25:.4f} (1 = perfect, 0 = random)")
print(f"\n{'Bin':>4} {'Predicted':>12} {'Observed':>12} {'Count':>8} {'Gap':>8}")
print("-" * 48)
for i in range(len(cal_predicted)):
    gap = abs(cal_predicted[i] - cal_observed[i])
    print(
        f"{i+1:>4} {cal_predicted[i]:>12.4f} {cal_observed[i]:>12.4f} "
        f"{cal_counts[i]:>8,} {gap:>8.4f}"
    )



### Checkpoint 1


In [ ]:
assert 0 <= brier <= 1, "Brier score must be between 0 and 1"
assert brier < 0.25, "Model should beat random (Brier < 0.25)"
print("\n[ok] Checkpoint 1 passed — calibration assessed\n")



## TASK 3 — TRAIN: one-way ANOVA + Tukey HSD


In [ ]:
print(f"\n=== One-Way ANOVA: Resale Price by Flat Type ===")

flat_types = ["2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE"]
anova_groups = []
anova_labels = []

for ft in flat_types:
    group = (
        hdb_recent.filter(pl.col("flat_type") == ft)["resale_price"]
        .to_numpy()
        .astype(np.float64)
    )
    if len(group) > 10:
        anova_groups.append(group)
        anova_labels.append(ft)
        print(
            f"  {ft:<12}: n={len(group):>6,}, "
            f"mean=${group.mean():>10,.0f}, std=${group.std():>8,.0f}"
        )

# TODO: Run one-way ANOVA across the flat-type groups.
# Hint: stats.f_oneway(*anova_groups) returns (f_statistic, p_value).
f_anova, p_anova = ____

print(f"\nANOVA F-statistic: {f_anova:.2f}")
print(f"p-value: {p_anova:.2e}")
print(
    f"{'SIGNIFICANT' if p_anova < 0.05 else 'NOT significant'}: "
    f"{'at least one flat type has a different mean price' if p_anova < 0.05 else 'no evidence of difference'}"
)

# TODO: Compute eta-squared (effect size).
# Hint: eta_sq = SS_between / SS_total where
#   SS_between = sum(n_i * (mean_i - grand_mean)^2)
#   SS_total   = sum((x_ij - grand_mean)^2)
all_data = np.concatenate(anova_groups)
grand_mean = all_data.mean()
ss_between = ____
ss_total = ____
eta_squared = ss_between / ss_total
print(
    f"Effect size (eta-sq): {eta_squared:.4f} "
    f"({eta_squared:.1%} of variance explained by flat type)"
)

# Post-hoc: Tukey HSD (Bonferroni-corrected pairwise t-tests)
print(f"\n--- Tukey HSD Post-Hoc Comparisons ---")
n_all = len(all_data)
k_groups = len(anova_groups)
ms_within = sum(np.sum((g - g.mean()) ** 2) for g in anova_groups) / (n_all - k_groups)

print(
    f"{'Comparison':<25} {'Diff ($)':>12} {'SE':>10} "
    f"{'q':>8} {'p-value':>10} {'Sig':>6}"
)
print("-" * 75)
for i in range(k_groups):
    for j in range(i + 1, k_groups):
        diff = anova_groups[j].mean() - anova_groups[i].mean()
        se = np.sqrt(
            ms_within * (1 / len(anova_groups[i]) + 1 / len(anova_groups[j])) / 2
        )
        q_stat = abs(diff) / se
        n_comparisons = k_groups * (k_groups - 1) / 2
        t_stat = diff / (
            np.sqrt(ms_within)
            * np.sqrt(1 / len(anova_groups[i]) + 1 / len(anova_groups[j]))
        )
        p_pair = 2 * (1 - stats.t.cdf(abs(t_stat), df=n_all - k_groups))
        # TODO: Apply Bonferroni correction to the pairwise p-value.
        # Hint: p_bonf = min(p_pair * n_comparisons, 1.0).
        p_bonf = ____
        sig = (
            "***"
            if p_bonf < 0.001
            else "**" if p_bonf < 0.01 else "*" if p_bonf < 0.05 else "ns"
        )
        label = f"{anova_labels[i]} vs {anova_labels[j]}"
        print(
            f"{label:<25} ${diff:>10,.0f} {se:>10,.0f} "
            f"{q_stat:>8.2f} {p_bonf:>10.4f} {sig:>6}"
        )



### Checkpoint 2


In [ ]:
assert f_anova > 0, "F-statistic must be positive"
assert 0 <= eta_squared <= 1, "Eta-squared must be between 0 and 1"
print("\n[ok] Checkpoint 2 passed — ANOVA + Tukey HSD completed\n")



## TASK 4 — VISUALISE: calibration plot + ANOVA box plots


In [ ]:
# Plot 1: Calibration curve
fig1 = go.Figure()
fig1.add_trace(
    go.Scatter(
        x=cal_predicted,
        y=cal_observed,
        mode="markers+lines",
        name="Model",
    )
)
fig1.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        name="Perfect calibration",
        line={"dash": "dash", "color": "grey"},
    )
)
fig1.update_layout(
    title=f"Calibration Curve (Brier={brier:.4f})",
    xaxis_title="Predicted Probability",
    yaxis_title="Observed Frequency",
)
fig1.write_html(str(OUTPUT_DIR / "calibration.html"))
print(f"Saved: {OUTPUT_DIR / 'calibration.html'}")

# TODO: Create ANOVA box plots for each flat type.
# Hint: loop over anova_labels and anova_groups, add go.Box traces.
fig2 = go.Figure()
for label, group in zip(anova_labels, anova_groups):
    fig2.add_trace(go.Box(y=group[:5000], name=label))
fig2.update_layout(
    title=f"Resale Price by Flat Type (ANOVA F={f_anova:.0f}, p<0.001)",
    yaxis_title="Resale Price ($)",
    xaxis_title="Flat Type",
)
fig2.write_html(str(OUTPUT_DIR / "anova_boxplot.html"))
print(f"Saved: {OUTPUT_DIR / 'anova_boxplot.html'}")

# Plot 3: Variance decomposition
fig3 = go.Figure()
fig3.add_trace(
    go.Bar(
        x=["Flat Type (between)", "Within groups"],
        y=[eta_squared, 1 - eta_squared],
        marker_color=["#3498db", "#bdc3c7"],
    )
)
fig3.update_layout(
    title=f"Variance Decomposition (eta-sq = {eta_squared:.3f})",
    yaxis_title="Proportion of Total Variance",
)
fig3.write_html(str(OUTPUT_DIR / "variance_decomposition.html"))
print(f"Saved: {OUTPUT_DIR / 'variance_decomposition.html'}")



### Checkpoint 3


In [ ]:
print("\n[ok] Checkpoint 3 passed — calibration + ANOVA visualised\n")



## TASK 5 — APPLY: HDB flat-type pricing policy


In [ ]:
# SCENARIO: HDB reviews its resale pricing guidelines. ANOVA tells us
# how much price variance is explained by flat type alone.

print(f"\n=== Real-World Application: HDB Pricing Policy ===")
print(f"\n  Variance explained by flat type: {eta_squared:.1%}")
if eta_squared > 0.3:
    print(f"  HIGH — flat type is a dominant price driver")
elif eta_squared > 0.1:
    print(f"  MODERATE — flat type matters but is not sufficient alone")
else:
    print(f"  LOW — other factors dominate price variation")

# Price differentials from baseline
baseline_type = "3 ROOM"
if baseline_type in anova_labels:
    baseline_idx = anova_labels.index(baseline_type)
    baseline_mean = anova_groups[baseline_idx].mean()
    print(f"\n  Price differentials vs {baseline_type} (${baseline_mean:,.0f}):")
    for label, group in zip(anova_labels, anova_groups):
        diff = group.mean() - baseline_mean
        pct = diff / baseline_mean * 100
        print(f"    {label:<12}: ${group.mean():>10,.0f}  ({pct:+.1f}%)")

print(f"\n  Model calibration (Brier): {brier:.4f}")
print(f"  Brier skill score: {1 - brier / 0.25:.3f}")
if brier < 0.15:
    print(f"  Well-calibrated — probabilities can inform premium pricing")
else:
    print(f"  Needs recalibration before using probabilities in pricing")



## REFLECTION


[x] Sigmoid: numerically stable implementation, verified properties
  [x] Logistic regression from scratch: Bernoulli MLE with gradient
  [x] sklearn comparison: validated implementation correctness
  [x] Odds ratios: exp(beta) = multiplicative change in odds per unit
  [x] Threshold optimisation: cost matrix drives the decision boundary
  [x] Confusion matrix: TP, FP, TN, FN and derived metrics
  [x] Precision, recall, F1: trade-offs in classification
  [x] ROC curve + AUC: discrimination across all thresholds
  [x] Precision-recall curve: better for imbalanced classes
  [x] Calibration: predicted probabilities match observed frequencies
  [x] Brier score: overall measure of probabilistic accuracy
  [x] One-way ANOVA: F-test for 3+ group means, eta-squared effect size
  [x] Tukey HSD: pairwise comparisons with multiple testing correction

  KEY INSIGHT: Classification metrics are not just academic — the
  CHOICE of metric directly determines business outcomes. Accuracy
  hides class imbalance; F1 balances precision and recall; the cost
  matrix encodes what your organisation actually loses from each
  type of error.

  Next: Exercise 7 — CUPED variance reduction for A/B tests, Bayesian
  A/B testing with posterior probabilities, sequential testing with
  always-valid p-values, and Difference-in-Differences for when
  randomisation is impossible.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print("\n[ok] Exercise 6 complete — Logistic Regression and Classification")

